In [2]:
print("Test")

Test


In [3]:
# Imports
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# Configurations
seed = 42

# Paths
DATA_DIR = pathlib.Path("../data/")
ENC2017_ROOT = DATA_DIR / "enc2017"
UD_ET_EDT_ROOT = DATA_DIR / "ud_et_edt"
HOMONYMS_ROOT = DATA_DIR / "homonymous_word_forms"

ENC2017_DIRS = {
    "processed": ENC2017_ROOT / "processed",
    "raw": ENC2017_ROOT / "raw",
}

UD_ET_EDT_DIRS = {
    "processed": UD_ET_EDT_ROOT / "processed",
    "raw": UD_ET_EDT_ROOT / "raw",
}

HOMONYMS_DIRS = {
    "processed": HOMONYMS_ROOT / "processed",
    "annotations": HOMONYMS_ROOT / "annotations",
}

OUTPUT_DIR = pathlib.Path("../outputs/")

MODEL_DIR = pathlib.Path("../models/")

In [5]:
import pandas as pd
import ast
import tqdm
import estnltk
from estnltk.default_resolver import make_resolver
from estnltk.taggers import VabamorfAnalyzer

from typing import Any

import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer, pipeline

### BERT MLM setup for TOP-k masked predictions

This section initialises a masked-language-model (MLM) BERT pipeline and provides a helper function for collecting TOP-10 predictions for a single `[MASK]` position.


In [6]:
# You can switch this to a local model directory if needed.
# Example: MODEL_NAME = "../models/your_mlm_model_directory"
MODEL_NAME = "EMBEDDIA/est-roberta"

# Device setup: GPU when available, otherwise CPU.
DEVICE = 0 if torch.cuda.is_available() else -1
print(f"Using model: {MODEL_NAME}")
print("Using device:", "cuda" if DEVICE == 0 else "cpu")

# Initialise tokenizer and MLM model.
mlm_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

# High-level fill-mask pipeline for easy TOP-k extraction.
fill_mask = pipeline(
    task="fill-mask",
    model=mlm_model,
    tokenizer=mlm_tokenizer,
    device=DEVICE,
    top_k=10,
    framework="pt",
)

# BERT-style models use [MASK] as the masking token.
MASK_TOKEN = mlm_tokenizer.mask_token
print("Mask token:", MASK_TOKEN)

Using model: EMBEDDIA/est-roberta
Using device: cuda


Device set to use cuda:0


Mask token: <mask>


In [26]:
# Examples of methods for doing a morphological analysis
# Using the VabamorfAnalyzer from estnltk
morph_analyzer = VabamorfAnalyzer()

example_text = "Linn on ilus."
estnltk_text_object = estnltk.Text(example_text)
estnltk_text_object.tag_layer(["words", "sentences"])
morph_analyzer.tag(estnltk_text_object)
print("Example morphological analysis with VabamorfAnalyzer:")
display(estnltk_text_object.morph_analysis)

# Usual method
estnltk_text_object = estnltk.Text(example_text)
estnltk_text_object.tag_layer("morph_analysis")
print("Example morphological analysis (usual method):")
display(estnltk_text_object.morph_analysis)

# Using a resolver
resolver = make_resolver()
estnltk_text_object = estnltk.Text(example_text)
estnltk_text_object.tag_layer(resolver=resolver)
print("Example morphological analysis (with resolver):")
display(estnltk_text_object.morph_analysis)

Example morphological analysis with VabamorfAnalyzer:


Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Linn', [{'normalized_text': 'Linn', 'lemma': 'Linn', 'root': 'Linn', 'root_tokens': ['Linn'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'H'}, {'normalized_text': 'Linn', 'lemma': 'linn', 'root': 'linn', 'root_tokens': ['linn'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('on', [{'normalized_text': 'on', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'b', 'partofspeech': 'V'}, {'normalized_text': 'on', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'vad', 'partofspeech': 'V'}]),
Span('ilus', [{'normalized_text': 'ilus', 'lemma': 'ilu', 'root': 'ilu', 'root_tokens': ['ilu'], 'ending': 's', 'clitic': '', 'form': 'sg in', 'partofspeech': 'S'}, {'normalized_text': 'ilus', 'lemma': 'ilus', 'root': 'ilus', 'root_tokens': ['ilus'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

Example morphological analysis (usual method):


Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Linn', [{'normalized_text': 'Linn', 'lemma': 'linn', 'root': 'linn', 'root_tokens': ['linn'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('on', [{'normalized_text': 'on', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'b', 'partofspeech': 'V'}, {'normalized_text': 'on', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'vad', 'partofspeech': 'V'}]),
Span('ilus', [{'normalized_text': 'ilus', 'lemma': 'ilus', 'root': 'ilus', 'root_tokens': ['ilus'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

Example morphological analysis (with resolver):


Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Linn', [{'normalized_text': 'Linn', 'lemma': 'linn', 'root': 'linn', 'root_tokens': ['linn'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('on', [{'normalized_text': 'on', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'b', 'partofspeech': 'V'}, {'normalized_text': 'on', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': '0', 'clitic': '', 'form': 'vad', 'partofspeech': 'V'}]),
Span('ilus', [{'normalized_text': 'ilus', 'lemma': 'ilus', 'root': 'ilus', 'root_tokens': ['ilus'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

In [ ]:
def prepare_mlm_input(text: str, masked_word_span: tuple[int, int]) -> str:
    """
    Prepare input text for masked language modeling by replacing the specified word span with a mask token.
    Args:
        text: The original input text.
        masked_word_span: A tuple (start_index, end_index) indicating the character span of the word to mask.
    Returns:
        A new string with the specified span replaced by the mask token.
    Raises:
        ValueError: If the masked_word_span is invalid (e.g., out of bounds or start >= end).
    """
    start, end = masked_word_span
    if start < 0 or end > len(text) or start >= end:
        raise ValueError(
            f"Invalid masked_word_span: {masked_word_span}. Must be within text bounds and start < end."
        )
    return text[:start] + MASK_TOKEN + text[end:]


def get_topk_mask_predictions(
    masked_text: str,
    top_k: int = 10,
    predictor: Any | None = None,
) -> list[dict[str, Any]]:
    """Return TOP-k predictions for one masked position.

    Args:
        masked_text: Input text that must contain exactly one mask token, e.g. `[MASK]`.
        top_k: Number of predictions to return.
        predictor: Optional fill-mask pipeline instance. Uses global `fill_mask` when None.

    Returns:
        A list of dictionaries with keys: rank, token, score, sequence.
    """
    model_predictor = predictor or fill_mask

    if masked_text.count(MASK_TOKEN) != 1:
        raise ValueError(
            f"Input must contain exactly one `{MASK_TOKEN}` token. Received: {masked_text}"
        )

    raw_predictions = model_predictor(masked_text, top_k=top_k)
    formatted_predictions: list[dict[str, Any]] = []

    for rank, prediction in enumerate(raw_predictions, start=1):
        token = prediction.get("token_str", "").strip()
        formatted_predictions.append(
            {
                "rank": rank,
                "token": token,
                "score": float(prediction["score"]),
                "sequence": prediction["sequence"],
                "morph_analysis": morph_analyzer.analyze_token(token),
            }
        )

    return formatted_predictions


def print_topk_predictions(
    text: str, predictions: list[dict[str, Any]], show_morph_analysis: bool = True
) -> None:
    """Print TOP-k MLM predictions in a readable table-like format.

    Args:
        text: The original masked input text.
        predictions: Output from `get_topk_mask_predictions`.
        show_morph_analysis: Whether to print morphological analysis for each predicted token.

    Returns:
        None.
    """
    print("Input text")
    print(f"  {text}")
    print("=" * 80)
    print(f"{'Rank':<6}{'Token':<20}{'Score':<12}Sequence")
    print("=" * 80)
    for item in predictions:
        # Print the main prediction info.
        print(
            f"{item['rank']:<6}{item['token']:<20}{item['score']:<12.6f}{item['sequence']}"
        )
        print("-" * 80)
        if show_morph_analysis:
            print("  Morph analysis for token:")
            print(f"    {'lemma':<20}{'root':<20}{'pos':<5}{'form':<10}")
            # Check all analyses for the token (some tokens may have multiple analyses).
            # and print them in a structured way.
            for analysis in item["morph_analysis"]:
                print(
                    f"    {analysis['lemma']:<20}{analysis['root']:<20}{analysis['partofspeech']:<5}{analysis['form']:<10}"
                )
            print("-" * 80)


example_text = f"{MASK_TOKEN} muna pole ma varem näinud."
top10 = get_topk_mask_predictions(example_text, top_k=10)
print_topk_predictions(example_text, top10)

Input text
  <mask> muna pole ma varem näinud.
Rank  Token               Score       Sequence
1     seda                0.065202    seda muna pole ma varem näinud.
--------------------------------------------------------------------------------
  Morph analysis for token:
    lemma               root                pos  form      
    see                 see                 P    sg p      
--------------------------------------------------------------------------------
2     Seda                0.058205    Seda muna pole ma varem näinud.
--------------------------------------------------------------------------------
  Morph analysis for token:
    lemma               root                pos  form      
    see                 see                 P    sg p      
--------------------------------------------------------------------------------
3     musta               0.043120    musta muna pole ma varem näinud.
---------------------------------------------------------------------------

In [93]:
homonyms_df = pd.read_csv(HOMONYMS_DIRS["processed"] / "homonyms_sentences_overall.csv")
display(homonyms_df.head())

,num,inflection_type,sentence,word,word_span,label,source
0,annotations,1,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"(74, 80)",['sg n'],../data/homonymous_word_forms/annotations/1/infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
1,annotations,1,"Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.",teooria,"(20, 27)",['sg n'],../data/homonymous_word_forms/annotations/1/infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
2,annotations,1,"""Ehk oleks mõttekas ka mõni selleteemaline hoiatav kampaania korraldada,"" lisab punase autoga preili.",kampaania,"(51, 60)",['sg n'],../data/homonymous_word_forms/annotations/1/infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
3,annotations,1,"""Minu otsus oli õige ning teeksin kõik sama moodi, kui saaksin uuesti teha,"" kommenteerib kolm aastat tagasi eriliste teenete eest Eesti passi saanud Primakov.",õige,"(16, 20)",['sg n'],../data/homonymous_word_forms/annotations/1/infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
4,annotations,1,"Itaalia president ütles Venemaa riigipea auks korraldatud suurejoonelisel banketil, et kahe riigi ühisavaldus Iraagi kohta oli kahe riigipea ""suur tarkuseavaldus"".",Itaalia,"(0, 7)",['sg g'],../data/homonymous_word_forms/annotations/1/infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


In [94]:
print(homonyms_df.head())

           num  inflection_type  \
0  annotations                1   
1  annotations                1   
2  annotations                1   
3  annotations                1   
4  annotations                1   

                                                                                                                                                                                                              sentence  \
0  Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.   
1                                                                               Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.   
2                                                                                                                "Ehk oleks mõttekas ka

In [95]:
example_row = homonyms_df.iloc[0]
example_sentence = example_row["sentence"]
example_masked_word = example_row["word"]
example_word_span = ast.literal_eval(example_row["word_span"])
print("Original sentence:", example_sentence)
print("Word span to mask:", example_word_span)
print("To be masked word:", example_masked_word)
masked_sentence = prepare_mlm_input(example_sentence, example_word_span)
print("Masked sentence:", masked_sentence)

Original sentence: Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.
Word span to mask: (74, 80)
To be masked word: võitja
Masked sentence: Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri <mask> James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.


In [107]:
sentences_with_predictions = {}
for index, row in tqdm.tqdm(homonyms_df.iterrows(), total=len(homonyms_df)):
    sentence = row["sentence"]
    form = ast.literal_eval(row["label"])[0]
    word_span = ast.literal_eval(row["word_span"])
    word = row["word"]
    inflection_type = row["inflection_type"]
    masked_sentence = prepare_mlm_input(sentence, word_span)
    top_predictions = get_topk_mask_predictions(masked_sentence, top_k=10)
    # print(f"Sentence: \n  {sentence}")
    # print("Matched form to predict:", form)
    # print("Matching predictions:")
    row_data = {
        "sentence": sentence,
        "word_to_mask": word,
        "word_span": word_span,
        "masked_sentence": masked_sentence,
        "form_to_predict": form,
        "inflection_type": inflection_type,
    }
    matching_predictions = []
    for pred in top_predictions:
        mas = pred["morph_analysis"]
        token = pred["token"]
        for ma in mas:
            if ma.get("form") == form and token != word:
                # print(f"  Predicted token: {pred['token']}")
                # print(f"    Morph analysis: {ma}")
                matching_predictions.append(pred)
    sentences_with_predictions[index] = {
        "row_data": row_data,
        "matching_predictions": matching_predictions,
    }
    break

  0%|          | 0/7886 [00:00<?, ?it/s]


In [120]:
for index, data in sentences_with_predictions.items():
    row_data = data["row_data"]
    matching_predictions = data["matching_predictions"]
    print(f"Sentence: \n  {row_data['sentence']}")
    print("Matched form to predict:", row_data["form_to_predict"])
    print("Matching predictions:")
    for pred in matching_predictions:
        print(f"  Predicted token: {pred['token']}")
        print(f"    Morph analysis: {pred['morph_analysis']}")

Sentence: 
  Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.
Matched form to predict: sg n
Matching predictions:
  Predicted token: võitnud
    Morph analysis: [{'root': 'võit=nud', 'root_tokens': ['võitnud'], 'ending': '0', 'clitic': '', 'partofspeech': 'A', 'form': '', 'lemma': 'võitnud'}, {'root': 'võit=nud', 'root_tokens': ['võitnud'], 'ending': '0', 'clitic': '', 'partofspeech': 'A', 'form': 'sg n', 'lemma': 'võitnud'}, {'root': 'võit=nu', 'root_tokens': ['võitnu'], 'ending': 'd', 'clitic': '', 'partofspeech': 'S', 'form': 'pl n', 'lemma': 'võitnu'}, {'root': 'võit=nud', 'root_tokens': ['võitnud'], 'ending': 'd', 'clitic': '', 'partofspeech': 'A', 'form': 'pl n', 'lemma': 'võitnud'}, {'root': 'võit', 'root_tokens': ['võit'], 'ending': 'nud', 'clitic': '', 'partofspeech': 'V', 'form': 'nud', 'lemma': 'võitma'}]
  Predict

In [ ]:
# Check if any sentence doesn't have any matching predictions.
no_match_count = sum(
    1
    for item in sentences_with_predictions.values()
    if not item["matching_predictions"]
)
print(
    f"Number of sentences with no matching predictions: {no_match_count} out of {len(sentences_with_predictions)}"
)

### Resolver-based TOP-10 MLM filtering pipeline

Pipeline logic (simple criterion):

- For each dataset row, build masked sentence from `word_span`.
- Get TOP-10 MLM token predictions for the mask.
- For each predicted token, create a candidate sentence by replacing the original span.
- Run EstNLTK resolver-based morph analysis on the whole candidate sentence.
- Keep only predictions where resolved `form` at the target token span matches the row's true form label.
- Save outputs in two files:
  - row-level summary table (one row per sentence),
  - matched-predictions table (one row per matching prediction).


In [ ]:
import json
from collections.abc import Iterable


def parse_span(span_value: object) -> tuple[int, int]:
    """Parse a span value into a `(start, end)` integer tuple.

    Args:
        span_value: Span in tuple form or string form like `"(74, 80)"`.

    Returns:
        Parsed `(start, end)` tuple.

    Raises:
        ValueError: If the input cannot be parsed into two integers.
    """
    if isinstance(span_value, tuple) and len(span_value) == 2:
        return (int(span_value[0]), int(span_value[1]))
    if isinstance(span_value, str):
        parsed = ast.literal_eval(span_value)
        if isinstance(parsed, tuple) and len(parsed) == 2:
            return (int(parsed[0]), int(parsed[1]))
    raise ValueError(f"Invalid span value: {span_value}")


def parse_label_form(label_value: object) -> str:
    """Extract the true form label from dataset label column.

    Args:
        label_value: Label value in string-list form (e.g. `"['sg n']"`) or iterable.

    Returns:
        A single form label string (e.g. `"sg n"`).

    Raises:
        ValueError: If no form label can be extracted.
    """
    if isinstance(label_value, str):
        parsed = ast.literal_eval(label_value)
        if isinstance(parsed, list) and parsed:
            return str(parsed[0]).strip()
        return str(label_value).strip()
    if isinstance(label_value, Iterable):
        as_list = list(label_value)
        if as_list:
            return str(as_list[0]).strip()
    raise ValueError(f"Invalid label value: {label_value}")


def sentence_with_token_replacement(
    sentence: str,
    original_span: tuple[int, int],
    replacement_token: str,
) -> tuple[str, tuple[int, int]]:
    """Replace the original word span with a predicted token and return new span.

    Args:
        sentence: Original sentence.
        original_span: Character span `(start, end)` for the original token.
        replacement_token: Predicted token to inject.

    Returns:
        A tuple of `(candidate_sentence, candidate_span)`.
    """
    start, end = original_span
    candidate_sentence = sentence[:start] + replacement_token + sentence[end:]
    candidate_span = (start, start + len(replacement_token))
    return candidate_sentence, candidate_span


def extract_labels_at_span_with_resolver(
    sentence: str,
    span: tuple[int, int],
    resolver_obj: Any,
    text_cache: dict[str, estnltk.Text],
) -> list[str]:
    """Resolve morph analysis on full sentence and extract labels for a target span.

    Args:
        sentence: Candidate sentence to analyse.
        span: Target character span `(start, end)` in candidate sentence.
        resolver_obj: EstNLTK resolver.
        text_cache: Cache mapping sentence string to tagged EstNLTK Text object.

    Returns:
        Sorted list of unique form values for analyses matching the target span.
    """
    if sentence not in text_cache:
        text_obj = estnltk.Text(sentence)
        text_obj.tag_layer(resolver=resolver_obj)
        text_cache[sentence] = text_obj

    text_obj = text_cache[sentence]
    labels = list()
    target_start, target_end = span

    for analysis in text_obj.morph_analysis:
        if analysis.start == target_start and analysis.end == target_end:
            form_value = analysis["form"]
            pos_value = analysis["partofspeech"]
            if isinstance(form_value, str):
                labels.append((str(form_value).strip(), str(pos).strip()))
            else:
                for form, pos in zip(form_value, pos_value):
                    labels.append((str(form).strip(), str(pos).strip()))

    return sorted(form for form in labels if form)


def run_mlm_form_filter_pipeline(
    df: pd.DataFrame,
    top_k: int = 10,
    save_outputs: bool = True,
    output_prefix: str = "bert_mlm_resolver",
) -> dict[str, Any]:
    """Run resolver-based TOP-k MLM filtering pipeline on homonym dataset.

    Args:
        df: Input dataframe with columns including sentence, word_span, label, word, inflection_type.
        top_k: Number of MLM predictions per row.
        save_outputs: Whether to save results to output files.
        output_prefix: Prefix for saved output files.

    Returns:
        Dictionary with `rows_df`, `matches_df`, and `results_by_index`.
    """
    resolver = make_resolver()
    text_cache: dict[str, estnltk.Text] = {}
    results_by_index: dict[int, dict[str, Any]] = {}

    row_records: list[dict[str, Any]] = []
    match_records: list[dict[str, Any]] = []

    for index, row in tqdm.tqdm(df.iterrows(), total=len(df)):
        sentence = str(row["sentence"]).strip()
        true_form = parse_label_form(row["label"]).strip()
        original_word = str(row["word"]).strip()
        inflection_type = row.get("inflection_type", None)
        original_span = parse_span(row["word_span"])
        masked_sentence = prepare_mlm_input(sentence, original_span)
        top_predictions = get_topk_mask_predictions(masked_sentence, top_k=top_k)

        matching_predictions: list[dict[str, Any]] = []

        for pred in top_predictions:
            token = str(pred["token"]).strip()
            if not token:
                continue

            candidate_sentence, candidate_span = sentence_with_token_replacement(
                sentence=sentence,
                original_span=original_span,
                replacement_token=token,
            )

            resolved_labels = extract_labels_at_span_with_resolver(
                sentence=candidate_sentence,
                span=candidate_span,
                resolver_obj=resolver,
                text_cache=text_cache,
            )

            # print(f"Token: {token}, Resolved forms at span: {resolved_labels}")
            # print(f"True form: {true_form}, Matches true form: {true_form in resolved_labels}")

            pred_record = {
                "rank": int(pred["rank"]),
                "token": token,
                "score": float(pred["score"]),
                "sequence": pred["sequence"],
                "candidate_sentence": candidate_sentence,
                "candidate_span": candidate_span,
                "resolved_forms": resolved_labels,
                "matches_true_form": true_form
                in [label[0] for label in resolved_labels],
            }

            if pred_record["matches_true_form"] and token != original_word:
                matching_predictions.append(pred_record)
                match_records.append(
                    {
                        "row_index": index,
                        "sentence": sentence,
                        "word_to_mask": original_word,
                        "word_span": original_span,
                        "masked_sentence": masked_sentence,
                        "true_form": true_form,
                        "inflection_type": inflection_type,
                        **pred_record,
                    }
                )

        row_record = {
            "row_index": index,
            "sentence": sentence,
            "word_to_mask": original_word,
            "word_span": original_span,
            "masked_sentence": masked_sentence,
            "true_form": true_form,
            "inflection_type": inflection_type,
            "top_k": top_k,
            "matched_predictions_count": len(matching_predictions),
            "has_match": len(matching_predictions) > 0,
        }
        row_records.append(row_record)

        results_by_index[index] = {
            "row_data": row_record,
            "matching_predictions": matching_predictions,
        }

    rows_df = pd.DataFrame(row_records)
    matches_df = pd.DataFrame(match_records)

    if save_outputs:
        rows_path = OUTPUT_DIR / f"{output_prefix}_rows.parquet"
        matches_path = OUTPUT_DIR / f"{output_prefix}_matches.parquet"
        nested_jsonl_path = OUTPUT_DIR / f"{output_prefix}_nested.jsonl"

        rows_df.to_parquet(rows_path, index=False)
        matches_df.to_parquet(matches_path, index=False)

        with open(nested_jsonl_path, "w", encoding="utf-8") as output_file:
            for _, item in results_by_index.items():
                output_file.write(
                    json.dumps(item, ensure_ascii=False, default=str) + "\n"
                )

        print("Saved outputs:")
        print(f"  Rows summary: {rows_path}")
        print(f"  Matched predictions: {matches_path}")
        print(f"  Nested JSONL: {nested_jsonl_path}")

    return {
        "rows_df": rows_df,
        "matches_df": matches_df,
        "results_by_index": results_by_index,
    }

In [ ]:
# Test the pipeline's functions
# Span parsing tests
assert parse_span("(74, 80)") == (74, 80)  # Test parse_span with string input
assert parse_span((74, 80)) == (74, 80)  # Test parse
# Form label parsing tests
assert parse_label_form("['sg n']") == "sg n"  # Test parse_label_form with string input
assert parse_label_form(["sg n"]) == "sg n"  # Test parse_label_form with
# Sentence with token replacement test
original_sentence = "Linn on ilus."
original_span = (0, 4)  # "Linn"
replacement_token = "Tartu"
candidate_sentence, candidate_span = sentence_with_token_replacement(
    sentence=original_sentence,
    original_span=original_span,
    replacement_token=replacement_token,
)
assert candidate_sentence == "Tartu on ilus."
assert candidate_span == (0, 5)
# Extract forms at span with resolver test
test_sentence = "Tartu on ilus."
test_span = (0, 5)  # "Tartu"
test_resolver = make_resolver()
test_text_cache = {}
resolved_labels = extract_labels_at_span_with_resolver(
    sentence=test_sentence,
    span=test_span,
    resolver_obj=test_resolver,
    text_cache=test_text_cache,
)
assert "sg n" in [
    label[0] for label in resolved_labels
]  # Check that "sg n" is among the resolved forms

In [132]:
sample_df = homonyms_df

In [133]:
# Run the pipeline on full dataset and store outputs.
pipeline_result = run_mlm_form_filter_pipeline(
    df=sample_df,
    top_k=10,
    save_outputs=True,
    output_prefix="bert_mlm_resolver_top10",
)

rows_df = pipeline_result["rows_df"]
matches_df = pipeline_result["matches_df"]

print("\nPipeline summary")
print("=" * 60)
print(f"Rows processed: {len(rows_df)}")
print(f"Rows with at least one matched prediction: {rows_df['has_match'].sum()}")
print(f"Rows without a match: {(~rows_df['has_match']).sum()}")
print(f"Total matched predictions: {len(matches_df)}")

display(rows_df.head())
display(matches_df.head())

100%|██████████| 7886/7886 [11:46<00:00, 11.16it/s]  


Saved outputs:
  Rows summary: ..\outputs\bert_mlm_resolver_top10_rows.parquet
  Matched predictions: ..\outputs\bert_mlm_resolver_top10_matches.parquet
  Nested JSONL: ..\outputs\bert_mlm_resolver_top10_nested.jsonl

Pipeline summary
Rows processed: 7886
Rows with at least one matched prediction: 7472
Rows without a match: 414
Total matched predictions: 45900


,row_index,sentence,word_to_mask,word_span,masked_sentence,true_form,inflection_type,top_k,matched_predictions_count,has_match
0,0,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"(74, 80)","Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri <mask> James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",sg n,1,10,4,True
1,1,"Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.",teooria,"(20, 27)","Normi-aktiveerimise <mask> (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.",sg n,1,10,8,True
2,2,"""Ehk oleks mõttekas ka mõni selleteemaline hoiatav kampaania korraldada,"" lisab punase autoga preili.",kampaania,"(51, 60)","""Ehk oleks mõttekas ka mõni selleteemaline hoiatav <mask> korraldada,"" lisab punase autoga preili.",sg n,1,10,9,True
3,3,"""Minu otsus oli õige ning teeksin kõik sama moodi, kui saaksin uuesti teha,"" kommenteerib kolm aastat tagasi eriliste teenete eest Eesti passi saanud Primakov.",õige,"(16, 20)","""Minu otsus oli <mask> ning teeksin kõik sama moodi, kui saaksin uuesti teha,"" kommenteerib kolm aastat tagasi eriliste teenete eest Eesti passi saanud Primakov.",sg n,1,10,9,True
4,4,"Itaalia president ütles Venemaa riigipea auks korraldatud suurejoonelisel banketil, et kahe riigi ühisavaldus Iraagi kohta oli kahe riigipea ""suur tarkuseavaldus"".",Itaalia,"(0, 7)","<mask> president ütles Venemaa riigipea auks korraldatud suurejoonelisel banketil, et kahe riigi ühisavaldus Iraagi kohta oli kahe riigipea ""suur tarkuseavaldus"".",sg g,1,10,9,True


,row_index,sentence,word_to_mask,word_span,masked_sentence,true_form,inflection_type,rank,token,score,sequence,candidate_sentence,candidate_span,resolved_forms,matches_true_form
0,0,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"(74, 80)","Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri <mask> James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",sg n,1,4,võitnud,0.051257,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitnud James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.","Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitnud James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.","(74, 81)","[(, A), (nud, V), (pl n, A), (sg n, A)]",True
1,0,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"(74, 80)","Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri <mask> James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",sg n,1,5,saanud,0.026861,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri saanud James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.","Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri saanud James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.","(74, 80)","[(, A), (nud, V), (pl n, A), (sg n, A)]",True
2,0,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"(74, 80)","Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri <mask> James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",sg n,1,7,pälvinud,0.015389,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri pälvinud James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.","Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri pälvinud James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.","(74, 82)","[(, A), (nud, V), (pl n, A), (sg n, A)]",True
3,0,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"(74, 80)","Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri <mask> James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",sg n,1,8,asutaja,0.009527,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri asutaja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.","Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri asutaja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.","(74, 81)","[(sg n, S)]"

In [ ]:
# Print rows without matches for manual inspection
print("\nRows without matches:")
no_match_rows = rows_df[~rows_df["has_match"]]
display(
    no_match_rows[
        ["masked_sentence", "word_to_mask", "true_form", "inflection_type"]
    ].head(10)
)
# Count without matches by inflection type and add percentages
no_match_by_inflection = (
    no_match_rows.groupby("inflection_type").size().reset_index(name="no_match_count")
)
total_by_inflection = (
    rows_df.groupby("inflection_type").size().reset_index(name="total_count")
)
merged_counts = pd.merge(
    no_match_by_inflection, total_by_inflection, on="inflection_type"
)
merged_counts["no_match_percentage"] = (
    merged_counts["no_match_count"] / merged_counts["total_count"]
) * 100
print("\nNo-match counts and percentages by inflection type:")
print(merged_counts)
# Count without matches by true form and add percentages
no_match_by_form = (
    no_match_rows.groupby("true_form").size().reset_index(name="no_match_count")
)
total_by_form = rows_df.groupby("true_form").size().reset_index(name="total_count")
merged_form_counts = pd.merge(no_match_by_form, total_by_form, on="true_form")
merged_form_counts["no_match_percentage"] = (
    merged_form_counts["no_match_count"] / merged_form_counts["total_count"]
) * 100
print("\nNo-match counts and percentages by true form:")
print(merged_form_counts.sort_values("no_match_percentage", ascending=False).head(20))


Rows without matches:


,masked_sentence,word_to_mask,true_form,inflection_type
19,"Põhja-Koreaga alustas Jaapan läbirääkimisi 1992. aastal, kuid need katkesid <mask> hiljem just samal põhjusel, mille üle nüüdki vaieldakse.",aasta,sg n,1
38,Andres <mask> Paetisme - Pühapäeval läheb Eesti suveajale.,Ülviste,sg g,1
52,"Laenu <mask> esitatud andmete õigsust kontrollib lisaks pangale laenu garanteerija Maaelu Laenude Tagamise Sihtasutus, mistõttu võib laenulepingu rahuldamine siiski veidi enam aega nõuda.",taotleja,sg g,1
57,"Näiteks on saatkondadena ära kadunud <mask>, Kanada ja Island, juurde aga tulnud Hispaania, Itaalia, Shveits, Slovakkia ja Jaapan.",Mehhiko,sg n,1
66,1985. <mask> pisikesest energiafirmast oli ta 15 aastaga kasvatanud oma turuväärtuse 1760 miljardile dollarile.,aasta,sg g,1
116,"Esikoha saanud meeskond pääseb jaanuari alguses Leipzigis peetavale <mask> kvalifikatsiooniturniirile, mille võitja saab Ateena olümpiale.",Euroopa,sg g,1
238,"Minu arvates on see väärt investeering, sest need riigid, kes <mask> Liidus meis kõige rohkem kahtlesid, osutusid meie sõpradeks tänu sellele, et nad said adekvaatset informatsiooni.",Euroopa,sg g,1
313,"Viimastel aastatel oli <mask> eestikeelseid saateid võimalik kuulda Vikerraadios, Klassikaraadios, Raadio Kukus, Raadio Kadis ning Tartu Raadios.",raadio,sg g,1
376,ETV-l tulnuks täiendavalt leida palgaks kuni <mask> miljonit krooni.,kaheksa,sg n,1
377,Kompromiss või bluff Tänaseks võivad valimas käinud kergendatult ohata - tänu linnapeale ja siseministrile <mask> võimuliit püsib.,Tartu,sg g,1



No-match counts and percentages by inflection type:
   inflection_type  no_match_count  total_count  no_match_percentage
0                1              55         1996             2.755511
1               16              80         1970             4.060914
2               17             181         1924             9.407484
3               19              98         1996             4.909820

No-match counts and percentages by true form:
  true_form  no_match_count  total_count  no_match_percentage
0       adt              30           94            31.914894
3      sg p              89          890            10.000000
2      sg n             128         2445             5.235174
1      sg g             167         4457             3.746915


In [155]:
display(no_match_rows[no_match_rows["true_form"] == "adt"].head())

,row_index,sentence,word_to_mask,word_span,masked_sentence,true_form,inflection_type,top_k,matched_predictions_count,has_match
2957,2957,"Siis sõideti Siguldasse, ja sedagi kord aastas, nüüd sõidetakse paari kuu tagant Sigtunasse ja Singapuri.",Singapuri,"(95, 104)","Siis sõideti Siguldasse, ja sedagi kord aastas, nüüd sõidetakse paari kuu tagant Sigtunasse ja <mask>.",adt,19,10,0,False
3002,3002,"""Ühe Euroopa sõidu ajal, kui arvati, et see idarahvas on natuke rumal, ja taheti presidendile tõlki, ütles Meri mikrofoni inglise keeles, et tal on vaja küsimusi, mitte tõlki!"" lausub 7. klassi poiss Tõnis.",mikrofoni,"(112, 121)","""Ühe Euroopa sõidu ajal, kui arvati, et see idarahvas on natuke rumal, ja taheti presidendile tõlki, ütles Meri <mask> inglise keeles, et tal on vaja küsimusi, mitte tõlki!"" lausub 7. klassi poiss Tõnis.",adt,19,10,0,False
3013,3013,"Samas on kõik uue kodu lähistel asuvad koolid sellised, mis võtavad gümnaasiumi vastu katsetega.",gümnaasiumi,"(68, 79)","Samas on kõik uue kodu lähistel asuvad koolid sellised, mis võtavad <mask> vastu katsetega.",adt,19,10,0,False
3087,3087,"Võite pommitamise lõpetada,"" kirjutas ta Belgradi tagasi saadetud teates.",Belgradi,"(41, 49)","Võite pommitamise lõpetada,"" kirjutas ta <mask> tagasi saadetud teates.",adt,19,10,0,False
3378,3378,"WTO pidi jaanuari alguses ütlema välja oma seisukoha geneetiliselt muundatud organismide küsimuses, kuid lükkas vastava arutelu veebruari.",veebruari,"(128, 137)","WTO pidi jaanuari alguses ütlema välja oma seisukoha geneetiliselt muundatud organismide küsimuses, kuid lükkas vastava arutelu <mask>.",adt,19,10,0,False


In [147]:
# Pretty format the rows without matches for easier reading
# and save to a text file for manual review.
no_match_output_path = OUTPUT_DIR / "bert_mlm_resolver_no_matches.txt"
with open(no_match_output_path, "w", encoding="utf-8") as no_match_file:
    for _, row in no_match_rows.iterrows():
        no_match_file.write(f"Masked sentence: {row['masked_sentence']}\n")
        no_match_file.write(f"Word to mask: {row['word_to_mask']}\n")
        no_match_file.write(f"True form: {row['true_form']}\n")
        no_match_file.write(f"Inflection type: {row['inflection_type']}\n")
        no_match_file.write("-" * 80 + "\n")